In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
d1 = pd.read_csv('gran/0_6_token_counts.csv', index_col = 'Unnamed: 0')
d2 = pd.read_csv('gran/6_12_token_counts.csv', index_col = 'Unnamed: 0')
d3 = pd.read_csv('gran/12_18_token_counts.csv', index_col = 'Unnamed: 0')
d4 = pd.read_csv('gran/18_24_token_counts.csv', index_col = 'Unnamed: 0')

In [2]:
d1.shape

(4000, 60)

In [3]:
df = pd.concat([d1,d2,d3,d4],axis=1)
df

,self_int_other_Send_small_No,self_ext_other_Send_small_Yes,Shared_Logoff,self_ext_self_View_small_Yes,other_ext_self_Send_small_Yes,other_disconnect_0,self_connect_4,self_int_other_View_small_No,self_ext_other_View_small_Yes,self_int_self_Send_small_Yes,...,other_connect_3,self_int_self_Send_large_Yes,self_ext_self_View_small_No,self_int_other_View_small_Yes,self_Visit_sus,self_connect_3,self_connect_2,Other_Logoff,other_External_Delete_,self_ext_other_View_large_Yes
DNS1758,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,9,0,0,551,0,0
ANC1950,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,26,0,0,0,0,0
SAB1954,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,3,0,0,0,0,0
LIM1718,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
LSN1672,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRN2617,0,0,0,0,0,0,0,0,0,0,...,0,5,0,22,59,0,0,0,0,7
VZG2484,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HVR0633,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CJS1986,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
import torch
import torch.nn as nn

# Encoder: BiLSTM + FNN
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, latent_dim):
        super(Encoder, self).__init__()
        self.bilstm1 = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.bilstm2 = nn.LSTM(hidden_dim*2, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.bilstm3 = nn.LSTM(hidden_dim*2, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),  # BiLSTM is bidirectional, so hidden_dim * 2
            nn.ReLU(),
            nn.Linear(128, latent_dim)  # Latent representation
        )

    def forward(self, x):
        x, _ = self.bilstm1(x)  # BiLSTM output
        x, _ = self.bilstm2(x)
        lstm_out, _ = self.bilstm3(x)
        latent = self.fc(lstm_out)  # Use last time step's output
        return latent

# Example parameters
input_dim = 60  # Features: x1, x2, x3, x4
hidden_dim = 64
num_layers = 2
latent_dim = 32

encoder = Encoder(input_dim, hidden_dim, num_layers, latent_dim)
print(encoder)


Encoder(
  (bilstm1): LSTM(60, 64, num_layers=2, batch_first=True, bidirectional=True)
  (bilstm2): LSTM(128, 64, num_layers=2, batch_first=True, bidirectional=True)
  (bilstm3): LSTM(128, 64, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=32, bias=True)
  )
)


In [5]:
# Decoder: FNN
class Decoder(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super(Decoder, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)  # Output dimension matches input dimension
        )

    def forward(self, x):
        reconstructed = self.fc(x)
        return reconstructed

# Example parameters
output_dim = input_dim * 1   # Reconstruct for all time steps (T)
decoder = Decoder(latent_dim, output_dim)
print(decoder)


Decoder(
  (fc): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=60, bias=True)
  )
)


In [6]:
class Autoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed


In [7]:
# Parameters
input_dim = 60    # Number of features per time step (e.g., 1 statistical value per 6h interval)
seq_len = 4      # Number of time steps (0-6h, 6-12h, 12-18h, 18-24h)
hidden_dim = 64  # BiLSTM hidden size
latent_dim = 32  # Latent space dimension

# Initialize components
encoder = Encoder(input_dim, hidden_dim, num_layers=2, latent_dim=latent_dim)
decoder = Decoder(latent_dim, input_dim)
autoencoder = Autoencoder(encoder, decoder)

print(autoencoder)


Autoencoder(
  (encoder): Encoder(
    (bilstm1): LSTM(60, 64, num_layers=2, batch_first=True, bidirectional=True)
    (bilstm2): LSTM(128, 64, num_layers=2, batch_first=True, bidirectional=True)
    (bilstm3): LSTM(128, 64, num_layers=2, batch_first=True, bidirectional=True)
    (fc): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=32, bias=True)
    )
  )
  (decoder): Decoder(
    (fc): Sequential(
      (0): Linear(in_features=32, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=60, bias=True)
    )
  )
)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder.to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)


In [9]:
# Simulated training data (replace with your dataset)
# Shape: [num_samples, seq_len=4, input_dim=1]
x_train = torch.tensor(d1.values.astype(np.float32))  # Ensure the data type is suitable for conversion.to(device)
train_loader = torch.utils.data.DataLoader(x_train, batch_size=1, shuffle=True)

# Training
epochs = 10
distances = []
for epoch in range(epochs):
    autoencoder.train()
    train_loss = 0

    for batch in train_loader:
        inputs = batch  # Input and target are the same for reconstruction
        
        optimizer.zero_grad()
        outputs = autoencoder(inputs)
        
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        distances.append(train_loss/len(train_loader))

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}")


RuntimeError: Input and parameter tensors are not at the same device, found input tensor at cpu and parameter tensor at cuda:0

In [ ]:
distances.sort()
distances

In [ ]:
distances[-5:]

In [ ]:
autoencoder.eval()

In [ ]:
d1.iloc[0].values, autoencoder(torch.tensor(d1.iloc[0].values.astype(np.float32).reshape([-1,1])).T)